In [ ]:
import math
import pandas as pd

from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForLanguageModeling, TrainingArguments, Trainer

In [ ]:
# Load datasets
data_path = Path("../Data/processed/")

data_files = {
    "train": str(data_path / "train.txt"),
    "validation": str(data_path / "valid.txt"),
    "test": str(data_path / "test.txt")
}

raw_datasets = load_dataset("text", data_files=data_files)

In [ ]:
# Load DistilGPT2
model_checkpoint = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForCausalLM.from_pretrained(model_checkpoint)

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

In [ ]:
# Tokenize and chunk

# Special token PHI 
special_tokens = {"additional_special_tokens": ["<PHI>"]}

num_added_tokens = tokenizer.add_special_tokens(special_tokens)

if num_added_tokens > 0:
    model.resize_token_embeddings(len(tokenizer))

# Tokenize datasets
def tokenize_function(examples):
    return tokenizer(examples["text"])

tokenized_datasets = raw_datasets.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

# Group tokens into fixed size blocks
block_size = 512

def group_texts(examples):
    concatenated_examples = {
        k: sum(examples[k], [])
        for k in examples.keys()
    }

    total_length = len(concatenated_examples["input_ids"])
    total_length = (total_length // block_size) * block_size

    result = {
        k: [
            t[i : i + block_size]
            for i in range(0, total_length, block_size)
        ]
        for k, t in concatenated_examples.items()
    }

    result["labels"] = result["input_ids"].copy()

    return result

lm_datasets = tokenized_datasets.map(
    group_texts,
    batched=True
)

lm_datasets

# Set up the data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
# Set training arguments
training_args = TrainingArguments(
    output_dir="../Models/",
    overwrite_output_dir=True,

    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    logging_steps=100,

    learning_rate=5e-5,
    weight_decay=0.01,
    num_train_epochs=3,
    max_steps=50,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=8,

    save_total_limit=2,
    load_best_model_at_end=True,

    report_to="none"
)

In [ ]:
# Create the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_datasets["train"],
    eval_dataset=lm_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator
)

In [ ]:
# Fine-tune model
trainer.train()

# Save model
trainer.save_model("../Models/distilgpt2-discharge-final")
tokenizer.save_pretrained("../Models/distilgpt2-discharge-final")

In [ ]:
# Evaluate on the test set
test_results = trainer.evaluate(lm_datasets["test"])
test_results

test_loss = test_results["eval_loss"]
perplexity = math.exp(test_loss)

print("Test loss:", test_loss)
print("Perplexity:", perplexity)